# **Features Selection - Xây dựng các đặc trưng cho mô hình**

Sau khi đã khám phá dữ liệu và những phương pháp huấn luyện mô hình đang có, trong phần này chúng tôi sẽ sàng lọc và cung cấp thêm những features quan trọng khác cho bộ dữ liệu huấn luyện của chúng tôi dựa trên đữ liệu cơ sở `base.csv`. Chúng tôi sẽ xây dựng tất cả các features có liên quan tới 2 phương pháp để dùng cho các bước sau

Cụ thể:
- Trích xuất toàn bộ dữ liệu chung hữu ích
- Xây dựng 2 bộ dữ liệu với 2 phương pháp khác nhau
- Tính toán, xây dựng các đặc trưng hữu ích cho từng bộ dữ liệu

**Mục tiêu**: Xây dựng lại tập huấn luyện với nhiều đặc trưng để cải thiện mô hình dự đoán

Mục lục:
1. [Thiết lập và cài đặt](#1)
2. [Xây dựng bộ dữ liệu cơ sở](#2)
3. [Dữ liệu cho dự đoán trực tiếp](#3)
4. [Dữ liệu cho dự đoán gián tiếp](#4)

<a id='1'></a>

## 1. Thiết lập và cài đặt

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys, os
sys.path.append(os.path.abspath('..'))
from src.get_data import get_connection, get_data_processed
from src.save_data import save_to_processed

In [2]:
conn = get_connection()

[OKE] Kết nối thành công tới database tại D:\datathon\vimchanhxa-datathon\data\database\datathon.duckdb


In [3]:
df_base = get_data_processed('base.csv')
df_base['date'] = pd.to_datetime(df_base['date'])
df_base.head()

Đã đọc thành công dữ liệu từ: D:\datathon\vimchanhxa-datathon\data\processed\base.csv


,date,revenue,cogs,is_test,order_count,unique_customers,total_quantity
0,2012-07-04,5123547.94,3982991.19,0,162.0,161.0,777.0
1,2012-07-05,2751773.45,2150580.23,0,97.0,97.0,428.0
2,2012-07-06,3054029.42,2517632.84,0,93.0,93.0,441.0
3,2012-07-07,2667930.94,2108246.62,0,73.0,73.0,364.0
4,2012-07-08,2360851.90,1808622.79,0,88.0,87.0,394.0


In [4]:
print(f"Train Period: {df_base[df_base['is_test'] == 0]['date'].min().date()} to {df_base[df_base['is_test'] == 0]['date'].max().date()}")
print(f"Test Period : {df_base[df_base['is_test'] == 1]['date'].min().date()} to {df_base[df_base['is_test'] == 1]['date'].max().date()}")

Train Period: 2012-07-04 to 2022-12-31
Test Period : 2023-01-01 to 2024-07-01


## 2. Xây dựng bộ feature

Trong phần này, chúng tôi sẽ trích xuất ra những features quan trọng, đặc biệt là thời gian và các dữ liệu tĩnh có tính lặp lại

### 2.1 Thời gian

In [5]:
df_base["days_since_start"] = (df_base["date"] - df_base["date"].min()).dt.days

**Merge với các feature đã được EDA**

In [6]:
df_sales_seasonal = pd.read_csv('../data/processed/sales_seasonal.csv')
df_sales_seasonal['date'] = pd.to_datetime(df_sales_seasonal['date'])

In [7]:
df_sales_seasonal.drop(columns=['revenue', 'cogs'], inplace=True)
df_base = df_base.merge(df_sales_seasonal, on=['date'], how='left')

In [8]:
df_base.columns

Index(['date', 'revenue', 'cogs', 'is_test', 'order_count', 'unique_customers',
       'total_quantity', 'days_since_start', 'month', 'day_of_month',
       'day_of_week', 'is_Sep_odd_year', 'is_Dec', 'is_Jun', 'seasonal_index',
       'day_of_week_index', 'intra_month_index', 'is_peak_season',
       'is_weekend', 'is_payday_spike', 'is_post_bill_slump', 'is_year_end',
       'revenue_lag_7', 'rm_7_missing', 'revenue_lag_364', 'rm_364_missing',
       'revenue_lag_28', 'rm_28_missing', 'rm_lag_28', 'rm_lag_364',
       'rm_lag_7'],
      dtype='str')

### 2.2 Sự kiện đặc biệt

In [9]:
# Sự kiện những ngày nhận lương, thường là từ 25 đến 5 hàng tháng, sẽ có nhu cầu mua sắm cao hơn
df_base["is_payday"] = np.where((df_base["day_of_month"] >= 25) | (df_base["day_of_month"] <= 5), 1, 0)

# Sự kiện những ngày có ngày trong tháng trùng với tháng (ví dụ 1/1, 2/2, ..., 12/12) có thể tạo ra những dịp mua sắm đặc biệt
# df_base["is_double_day"] = np.where(df_base["month"] == df_base["day_of_month"], 1, 0)

In [10]:
df_base.columns

Index(['date', 'revenue', 'cogs', 'is_test', 'order_count', 'unique_customers',
       'total_quantity', 'days_since_start', 'month', 'day_of_month',
       'day_of_week', 'is_Sep_odd_year', 'is_Dec', 'is_Jun', 'seasonal_index',
       'day_of_week_index', 'intra_month_index', 'is_peak_season',
       'is_weekend', 'is_payday_spike', 'is_post_bill_slump', 'is_year_end',
       'revenue_lag_7', 'rm_7_missing', 'revenue_lag_364', 'rm_364_missing',
       'revenue_lag_28', 'rm_28_missing', 'rm_lag_28', 'rm_lag_364',
       'rm_lag_7', 'is_payday'],
      dtype='str')

In [11]:
# # Những dịp tết âm lịch
# tet_dates = {
#     2012: "2012-01-23",
#     2013: "2013-02-10",
#     2014: "2014-01-31",
#     2015: "2015-02-19",
#     2016: "2016-02-08",
#     2017: "2017-01-28",
#     2018: "2018-02-16",
#     2019: "2019-02-05",
#     2020: "2020-01-25",
#     2021: "2021-02-12",
#     2022: "2022-02-01",
#     2023: "2023-01-22",
#     2024: "2024-02-10",
# }
# tet_map = pd.DataFrame(list(tet_dates.items()), columns=["year", "tet_date"])
# tet_map["tet_date"] = pd.to_datetime(tet_map["tet_date"])

# df_base = df_base.merge(tet_map, on="year", how="left")
# # df_base["days_to_tet"] = (df_base["tet_date"] - df_base["date"]).dt.days
# # df_base["is_pre_tet_rush"] = np.where((df_base["days_to_tet"] > 0) & (df_base["days_to_tet"] <= 21), 1, 0)
# # df_base["is_tet_holiday"] = np.where((df_base["days_to_tet"] <= 0) & (df_base["days_to_tet"] >= -6), 1, 0)
# df_base = df_base.drop(columns=["tet_date"])

Như ở phần `EDA`, chúng ta đã thấy rằng doanh nghiệp hiện tại đang mở các đợt giảm giá theo chu kỳ lặp lại hằng năm và hai năm. Cụ thể là với các mã giảm giá `Spring Sale`, `Mid-Year Sale`, `Fall Launch`, `Year-End Sale` đang được mở lại qua từng năm, còn với `Urban Blowout` và `Rural Special` thì lại lặp 2 năm một lần ( các năm lẻ ). Đặc biệt hơn là thời gian bắt đầu và kết thúc cũng rất khớp nhau, cùng lắm chỉ lệch 1 ngày. Tuy nhiên có một vài biến trong đó sẽ không được lặp lại giống hệt các năm trước như `min_order_value`, vì vậy dưới đây ta sẽ làm lại dữ liệu promotions này.

Do vẫn còn sự khác nhau, nên chúng tôi sẽ tạo thêm 2 cột đặc trưng để tránh bị loãng mô hình bao gồm: `is_promo` và `total_discount`, những ngày giảm giá và phần trăm đang giảm giá. Mục đích là hạn chế tạo quá nhiều cột nhiễu không cần thiết và vẫn thể hiện được sự khác biệt so với các mã giảm giá và các ngày khác. Sau đó, nếu `total_discount` không thể hiện được rõ vai trò, chúng tôi sẽ xóa đi vì trong dữ liệu sẽ có một vài mã có `stackable_flag` là không cho sử dụng 2 mã cùng lúc.

### 2.4 Mã giảm giá

In [12]:
promotions = conn.execute("SELECT * FROM promotions").df()
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

In [13]:
# Lọc các mã giảm giá lặp lại
promo_range = pd.date_range(start=df_base["date"].min(), end=df_base["date"].max())

daily_p = pd.DataFrame({
    "date": promo_range,
    "is_promo": 0,
    "total_discount": 0.0
})

for _, row in promotions.iterrows():
    mask = (
        (daily_p["date"] >= row["start_date"]) &
        (daily_p["date"] <= row["end_date"])
    )
    daily_p.loc[mask, "is_promo"] = 1
    daily_p.loc[mask, "total_discount"] += row["discount_value"]

daily_p.head()


,date,is_promo,total_discount
0,2012-07-04,0,0.0
1,2012-07-05,0,0.0
2,2012-07-06,0,0.0
3,2012-07-07,0,0.0
4,2012-07-08,0,0.0


In [14]:

# df_base = df_base.merge(daily_p, on="date", how="left").fillna({"is_promo": 0, "total_discount": 0})
df_base = df_base.merge(daily_p, on=['date'], how='left').fillna({"is_promo": 0, "total_discount": 0})
df_base.columns

Index(['date', 'revenue', 'cogs', 'is_test', 'order_count', 'unique_customers',
       'total_quantity', 'days_since_start', 'month', 'day_of_month',
       'day_of_week', 'is_Sep_odd_year', 'is_Dec', 'is_Jun', 'seasonal_index',
       'day_of_week_index', 'intra_month_index', 'is_peak_season',
       'is_weekend', 'is_payday_spike', 'is_post_bill_slump', 'is_year_end',
       'revenue_lag_7', 'rm_7_missing', 'revenue_lag_364', 'rm_364_missing',
       'revenue_lag_28', 'rm_28_missing', 'rm_lag_28', 'rm_lag_364',
       'rm_lag_7', 'is_payday', 'is_promo', 'total_discount'],
      dtype='str')

### 2.5 Traffic & CR

In [15]:
web_traffic = conn.execute('SELECT * FROM web_traffic').df()
web_traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.005,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.004,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.004,263.6,direct
3,2013-01-04,9973,8063,53078,0.006,151.8,direct
4,2013-01-05,10223,7882,36790,0.005,168.6,referral


In [16]:
# Join web_traffic với sales để lấy revenue
# Bước 1: Lấy dữ liệu sales
df_sales_join = conn.execute('SELECT date, revenue FROM sales').df()
df_sales_join['date'] = pd.to_datetime(df_sales_join['date'])

# Bước 2: Lấy dữ liệu web_traffic và chuẩn hóa tên cột
df_web = conn.execute('SELECT date, sessions, unique_visitors, page_views, bounce_rate, avg_session_duration_sec FROM web_traffic').df()
df_web['date'] = pd.to_datetime(df_web['date'])

# Chuẩn hóa tên cột (thống nhất unique_visitors và 'unique visitors')
df_web.columns = [col.strip().lower().replace(' ', '_') for col in df_web.columns]
print("Các cột trong web_traffic sau chuẩn hóa:")
print(df_web.columns.tolist())

# Bước 3: Join hai bảng theo date
df_traffic_merged = pd.merge(df_sales_join, df_web, on='date', how='outer')

# Bước 4: Thêm cờ missing values cho traffic
df_traffic_merged['missing_traffic'] = df_traffic_merged['sessions'].isna() | df_traffic_merged['unique_visitors'].isna()

print(f"\nKích thước dữ liệu sau khi join và loại bỏ 2012: {df_traffic_merged.shape}")
print(f"Năm trong dữ liệu: {sorted(df_traffic_merged['date'].dt.year.unique())}")
df_traffic_merged.head()

Các cột trong web_traffic sau chuẩn hóa:
['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec']

Kích thước dữ liệu sau khi join và loại bỏ 2012: (3833, 8)
Năm trong dữ liệu: [np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


,date,revenue,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,missing_traffic
0,2012-07-04,5123547.94,NaN,NaN,NaN,NaN,NaN,True
1,2012-07-05,2751773.45,NaN,NaN,NaN,NaN,NaN,True
2,2012-07-06,3054029.42,NaN,NaN,NaN,NaN,NaN,True
3,2012-07-07,2667930.94,NaN,NaN,NaN,NaN,NaN,True
4,2012-07-08,2360851.90,NaN,NaN,NaN,NaN,NaN,True


In [17]:
# Tính Conversion Rate từ order_count trong processed/base.csv và sessions trong web_traffic
# Bước 1: Đọc dữ liệu base.csv để lấy order_count
df_base_unmerged = pd.read_csv('../data/processed/base.csv')
df_base_unmerged['date'] = pd.to_datetime(df_base_unmerged['date'])

# Chỉ lấy cột date và order_count
df_orders = df_base_unmerged[['date', 'order_count']].copy()

# Bước 2: Lấy sessions từ web_traffic (đã có trong df_traffic_merged)
df_sessions = df_traffic_merged[['date', 'sessions']].copy()

# Bước 3: Merge để tính Conversion Rate theo từng date
df_cr = pd.merge(df_orders, df_sessions, on='date', how='inner')

# Bước 4: Tính Conversion Rate = order_count / sessions
df_cr['conversion_rate'] = df_cr['order_count'] / df_cr['sessions']

# Thay thế giá trị vô cùng hoặc NaN bằng 0
df_cr['conversion_rate'] = df_cr['conversion_rate'].replace([np.inf, -np.inf], 0).fillna(0)

# Bước 5: Gộp cột conversion_rate vào df_traffic_merged
df_traffic_merged = pd.merge(df_traffic_merged, df_cr[['date', 'conversion_rate']], on='date', how='left')
df_traffic_merged.drop(columns=['revenue'], inplace=True)
df_traffic_merged.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,missing_traffic,conversion_rate
0,2012-07-04,NaN,NaN,NaN,NaN,NaN,True,0.0
1,2012-07-05,NaN,NaN,NaN,NaN,NaN,True,0.0
2,2012-07-06,NaN,NaN,NaN,NaN,NaN,True,0.0
3,2012-07-07,NaN,NaN,NaN,NaN,NaN,True,0.0
4,2012-07-08,NaN,NaN,NaN,NaN,NaN,True,0.0


In [18]:
# Gộp vào df_base (chỉ lấy các cột cần thiết, loại bỏ revenue/cogs để tránh trùng lặp)
df_traffic_cols = ['date', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 
                   'avg_session_duration_sec', 'conversion_rate']
df_base = pd.merge(df_base, df_traffic_merged[df_traffic_cols], on='date', how='left')

# !!!!!!!!!!!!!!!!!!!!!!!!
# Fill missing values cho các cột từ web_traffic
# Các cột đếm được (sessions, visitors, page_views) -> 0
# Các cột tỷ lệ (bounce_rate, duration, conversion_rate) -> 0.0
traffic_count_cols = ['sessions', 'unique_visitors', 'page_views']
traffic_rate_cols = ['bounce_rate', 'avg_session_duration_sec', 'conversion_rate']

df_base[traffic_count_cols] = df_base[traffic_count_cols].fillna(0)
df_base[traffic_rate_cols] = df_base[traffic_rate_cols].fillna(0.0)

In [19]:
df_base.head()

,date,revenue,cogs,is_test,order_count,unique_customers,total_quantity,days_since_start,month,day_of_month,...,rm_lag_7,is_payday,is_promo,total_discount,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,conversion_rate
0,2012-07-04,5123547.94,3982991.19,0,162.0,161.0,777.0,0,7.0,4.0,...,NaN,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2012-07-05,2751773.45,2150580.23,0,97.0,97.0,428.0,1,7.0,5.0,...,NaN,1,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2012-07-06,3054029.42,2517632.84,0,93.0,93.0,441.0,2,7.0,6.0,...,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2012-07-07,2667930.94,2108246.62,0,73.0,73.0,364.0,3,7.0,7.0,...,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2012-07-08,2360851.90,1808622.79,0,88.0,87.0,394.0,4,7.0,8.0,...,NaN,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 2.6 Orders & AOV

In [20]:
# Tính AOV (Average Order Value) = revenue / order_count
# Lưu ý: revenue đã có trong df_base, không cần merge thêm
df_aov = df_base[['date', 'revenue', 'order_count']].copy()

# Tính AOV = revenue / order_count
df_aov['average_order_value'] = df_aov['revenue'] / df_aov['order_count']

# Thay thế giá trị vô cùng hoặc NaN bằng 0
df_aov['average_order_value'] = df_aov['average_order_value'].replace([np.inf, -np.inf], 0).fillna(0)

# Merge cột average_order_value vào df_base
df_base = pd.merge(df_base, df_aov[['date', 'average_order_value']], on='date', how='left')

df_base.columns

Index(['date', 'revenue', 'cogs', 'is_test', 'order_count', 'unique_customers',
       'total_quantity', 'days_since_start', 'month', 'day_of_month',
       'day_of_week', 'is_Sep_odd_year', 'is_Dec', 'is_Jun', 'seasonal_index',
       'day_of_week_index', 'intra_month_index', 'is_peak_season',
       'is_weekend', 'is_payday_spike', 'is_post_bill_slump', 'is_year_end',
       'revenue_lag_7', 'rm_7_missing', 'revenue_lag_364', 'rm_364_missing',
       'revenue_lag_28', 'rm_28_missing', 'rm_lag_28', 'rm_lag_364',
       'rm_lag_7', 'is_payday', 'is_promo', 'total_discount', 'sessions',
       'unique_visitors', 'page_views', 'bounce_rate',
       'avg_session_duration_sec', 'conversion_rate', 'average_order_value'],
      dtype='str')

### 2.7 Items & IPO

In [21]:
df_orders = df_base[['date', 'order_count', 'total_quantity']].copy()

# Bước 3: Merge để tính Conversion Rate theo từng date
df_ipo = df_orders.copy()

# Bước 4: Tính Conversion Rate = order_count / sessions
df_ipo['items_per_order'] = df_ipo['total_quantity'] / df_ipo['order_count']

# Thay thế giá trị vô cùng hoặc NaN bằng 0
df_ipo['items_per_order'] = df_ipo['items_per_order'].replace([np.inf, -np.inf], 0).fillna(0)

# Merge cột ipo vào df_base
df_base = pd.merge(df_base, df_ipo[['date', 'items_per_order']], on='date', how='left')

df_base.columns

Index(['date', 'revenue', 'cogs', 'is_test', 'order_count', 'unique_customers',
       'total_quantity', 'days_since_start', 'month', 'day_of_month',
       'day_of_week', 'is_Sep_odd_year', 'is_Dec', 'is_Jun', 'seasonal_index',
       'day_of_week_index', 'intra_month_index', 'is_peak_season',
       'is_weekend', 'is_payday_spike', 'is_post_bill_slump', 'is_year_end',
       'revenue_lag_7', 'rm_7_missing', 'revenue_lag_364', 'rm_364_missing',
       'revenue_lag_28', 'rm_28_missing', 'rm_lag_28', 'rm_lag_364',
       'rm_lag_7', 'is_payday', 'is_promo', 'total_discount', 'sessions',
       'unique_visitors', 'page_views', 'bounce_rate',
       'avg_session_duration_sec', 'conversion_rate', 'average_order_value',
       'items_per_order'],
      dtype='str')

## 3. Tổng kết:

### 3.1 Fill missing values tất cả các cột

In [22]:
### 3.1 Fill missing values tất cả các cột

# Fill tất cả missing values còn lại
# Các cột numeric cần fill: revenue, cogs, order_count, unique_customers, total_quantity
numeric_cols = ['revenue', 'cogs', 'order_count', 'unique_customers', 'total_quantity']
for col in numeric_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(0)

# Các cột lag/rolling: fill 0 hoặc forward fill tùy logic
lag_cols = ['revenue_lag_7', 'revenue_lag_364', 'revenue_lag_28', 'rm_lag_7', 'rm_lag_364', 'rm_lag_28']
for col in lag_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(0)

# Các cột missing flag: fill 1 (đánh dấu là có missing trong giai đoạn đầu)
missing_flag_cols = ['rm_7_missing', 'rm_364_missing', 'rm_28_missing']
for col in missing_flag_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(1)

# Các cột seasonal/index: fill 0
index_cols = ['seasonal_index', 'day_of_week_index', 'intra_month_index']
for col in index_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(0)

# Các cột binary: fill 0
binary_cols = ['is_Sep_odd_year', 'is_Dec', 'is_Jun', 'is_peak_season', 'is_weekend', 
               'is_payday_spike', 'is_post_bill_slump', 'is_year_end', 'is_promo']
for col in binary_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(0)

# Các cột tỷ lệ: fill 0.0
rate_cols = ['total_discount', 'bounce_rate', 'avg_session_duration_sec', 'conversion_rate', 
             'average_order_value', 'items_per_order']
for col in rate_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(0.0)

# Các cột traffic: convert sang float rồi fill 0
traffic_count_cols = ['sessions', 'unique_visitors', 'page_views']
for col in traffic_count_cols:
    if col in df_base.columns:
        df_base[col] = pd.to_numeric(df_base[col], errors='coerce').fillna(0)

# Các cột month/day: fill giá trị phổ biến nhất hoặc 0
time_cols = ['month', 'day_of_month', 'day_of_week']
for col in time_cols:
    if col in df_base.columns:
        df_base[col] = df_base[col].fillna(0)

print("Đã fill tất cả missing values!")
print(f"\nSố chiều của dữ liệu sau khi feature selection: {df_base.shape}")

Đã fill tất cả missing values!

Số chiều của dữ liệu sau khi feature selection: (4381, 42)


In [23]:
df_base.isnull().sum()

date                        0
revenue                     0
cogs                        0
is_test                     0
order_count                 0
unique_customers            0
total_quantity              0
days_since_start            0
month                       0
day_of_month                0
day_of_week                 0
is_Sep_odd_year             0
is_Dec                      0
is_Jun                      0
seasonal_index              0
day_of_week_index           0
intra_month_index           0
is_peak_season              0
is_weekend                  0
is_payday_spike             0
is_post_bill_slump          0
is_year_end                 0
revenue_lag_7               0
rm_7_missing                0
revenue_lag_364             0
rm_364_missing              0
revenue_lag_28              0
rm_28_missing               0
rm_lag_28                   0
rm_lag_364                  0
rm_lag_7                    0
is_payday                   0
is_promo                    0
total_disc

In [24]:
df_base.columns

Index(['date', 'revenue', 'cogs', 'is_test', 'order_count', 'unique_customers',
       'total_quantity', 'days_since_start', 'month', 'day_of_month',
       'day_of_week', 'is_Sep_odd_year', 'is_Dec', 'is_Jun', 'seasonal_index',
       'day_of_week_index', 'intra_month_index', 'is_peak_season',
       'is_weekend', 'is_payday_spike', 'is_post_bill_slump', 'is_year_end',
       'revenue_lag_7', 'rm_7_missing', 'revenue_lag_364', 'rm_364_missing',
       'revenue_lag_28', 'rm_28_missing', 'rm_lag_28', 'rm_lag_364',
       'rm_lag_7', 'is_payday', 'is_promo', 'total_discount', 'sessions',
       'unique_visitors', 'page_views', 'bounce_rate',
       'avg_session_duration_sec', 'conversion_rate', 'average_order_value',
       'items_per_order'],
      dtype='str')

In [25]:
save_to_processed(df_base, "final_features.csv")

Đã lưu thành công tại: D:\datathon\vimchanhxa-datathon\data\processed\final_features.csv


---
**Kết luận:**

---
**Notebooks tiếp theo:** [05_BASELINE_MODEL_.ipynb](05_BASELINE_MODEL_.ipynb) - Xây dựng mô hình cơ sở